In [1]:
import numpy as np
import pandas as pd
import re
from pathlib import Path

# =========================
# USER INPUTS
# =========================
URL_BOUNDS = "https://raw.githubusercontent.com/Raziye-Aghapour/ReEDS_Morris/refs/heads/main/all_multiplier_bounds.csv"
R_TRAJ = 5          # number of Morris trajectories
P_LEVELS = 6        # grid levels (recommended >= 6 so delta is not huge)
SEED = 123
OUTDIR = Path("morris_samples")
OUTDIR.mkdir(exist_ok=True)

# =========================
# 1) READ BOUNDS TABLE
# =========================
bounds_df = pd.read_csv(URL_BOUNDS)

# basic cleanup
bounds_df["Parameter"] = bounds_df["Parameter"].astype(str).str.strip()
bounds_df["Unit"] = bounds_df["Unit"].astype(str).str.strip()
bounds_df["Upperbound"] = pd.to_numeric(bounds_df["Upperbound"], errors="coerce")
bounds_df["Lowerbound"] = pd.to_numeric(bounds_df["Lowerbound"], errors="coerce")
bounds_df = bounds_df.dropna(subset=["Parameter", "Upperbound", "Lowerbound"]).reset_index(drop=True)

# =========================
# 2) MAKE SAFE FACTOR NAMES
# =========================
def slugify(name: str) -> str:
    s = name.strip().lower()
    s = s.replace("&", "and")
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

bounds_df["factor"] = bounds_df["Parameter"].apply(slugify)

# Ensure uniqueness in case two parameters slugify to the same string
if bounds_df["factor"].duplicated().any():
    counts = {}
    new = []
    for f in bounds_df["factor"]:
        counts[f] = counts.get(f, 0) + 1
        new.append(f"{f}__{counts[f]}")
    bounds_df["factor"] = new

# =========================
# 3) MORRIS TRAJECTORIES (SCALED TO BOUNDS)
# =========================
def morris_trajectories(bounds: pd.DataFrame, r: int, p: int, seed: int = 123):
    """
    Returns:
      design_wide: DataFrame with run_id, factor columns (scaled), trajectory, step, changed_factor
      factor_map:  DataFrame mapping factor -> original parameter + bounds + unit
    """
    rng = np.random.default_rng(seed)
    k = len(bounds)

    # p-level grid in [0,1]
    grid = np.linspace(0, 1, p)
    # standard Morris delta choice
    delta = p / (2 * (p - 1))

    # Starting points must allow a +/- delta move staying within [0,1]
    start_candidates = grid[grid <= (1 - delta + 1e-12)]
    if len(start_candidates) == 0:
        raise ValueError(
            f"No valid starting points for p={p} (delta={delta:.4f}). "
            "Increase p (e.g., 6, 8, 10)."
        )

    X_all = []
    meta_rows = []
    run_id = 0

    for traj in range(r):
        # Random base point on the allowable grid
        x = rng.choice(start_candidates, size=k, replace=True)

        # Random order to perturb factors
        order = rng.permutation(k)

        # Random direction (+/-) per factor
        directions = rng.choice([-1, 1], size=k)

        # First point in trajectory
        X_all.append(x.copy())
        meta_rows.append({"run_id": run_id, "trajectory": traj, "step": 0, "changed_factor": ""})
        run_id += 1

        # Step through each factor once (OAT)
        for step, j in enumerate(order, start=1):
            x_new = x.copy()
            d = directions[j] * delta

            # If out of bounds, flip direction
            if not (0 <= x_new[j] + d <= 1):
                d = -d

            x_new[j] = x_new[j] + d

            # Snap to nearest grid point (numerical safety)
            x_new[j] = grid[np.argmin(np.abs(grid - x_new[j]))]

            X_all.append(x_new.copy())
            meta_rows.append({
                "run_id": run_id,
                "trajectory": traj,
                "step": step,
                "changed_factor": bounds.loc[j, "factor"],
            })
            run_id += 1
            x = x_new

    # Normalize matrix [0,1]
    X = np.vstack(X_all)

    # Scale to actual bounds
    lo = bounds["Lowerbound"].to_numpy(dtype=float)
    hi = bounds["Upperbound"].to_numpy(dtype=float)
    X_scaled = lo + X * (hi - lo)

    meta = pd.DataFrame(meta_rows)

    design = pd.DataFrame(X_scaled, columns=bounds["factor"].tolist())
    design.insert(0, "run_id", meta["run_id"].values)
    design = pd.concat([design, meta[["trajectory", "step", "changed_factor"]]], axis=1)

    factor_map = bounds[["factor", "Parameter", "Lowerbound", "Upperbound", "Unit"]].copy()

    return design, factor_map, delta

design, factor_map, delta = morris_trajectories(bounds_df, r=R_TRAJ, p=P_LEVELS, seed=SEED)

# =========================
# 4) SAVE OUTPUTS
# =========================
design_path = OUTDIR / f"morris_design_r{R_TRAJ}_p{P_LEVELS}.csv"
map_path = OUTDIR / "morris_factor_map.csv"

design.to_csv(design_path, index=False)
factor_map.to_csv(map_path, index=False)

k = len(bounds_df)
n_runs = len(design)

print("=== Morris sampling created ===")
print(f"Factors (k): {k}")
print(f"Trajectories (r): {R_TRAJ}")
print(f"Grid levels (p): {P_LEVELS}")
print(f"Delta (normalized): {delta:.6f}")
print(f"Total runs: r*(k+1) = {R_TRAJ}*({k}+1) = {n_runs}")
print(f"Wrote: {design_path}")
print(f"Wrote: {map_path}")

# Optional preview
print("\nPreview (first 3 runs):")
print(design.head(3).iloc[:, :10])  # first few columns only


=== Morris sampling created ===
Factors (k): 39
Trajectories (r): 5
Grid levels (p): 6
Delta (normalized): 0.600000
Total runs: r*(k+1) = 5*(39+1) = 200
Wrote: morris_samples/morris_design_r5_p6.csv
Wrote: morris_samples/morris_factor_map.csv

Preview (first 3 runs):
   run_id  electricity_demand_multiplier_tx  \
0       0                            0.9579   
1       1                            0.9579   
2       2                            0.9579   

   natural_gas_price_west_south_central  battery_capcost  \
0                               1.11522          0.65902   
1                               1.11522          0.65902   
2                               1.11522          0.65902   

   battery_capcost_energy  coal_igcc_capcost  coal_igcc_fom  coal_igcc_vom  \
0                  0.8124            0.99442         0.9724         0.9724   
1                  0.8124            0.99442         0.9724         0.9724   
2                  0.8124            0.99442         0.9724         